In [ ]:
import jax
jax.config.update('jax_enable_x64', True)  # Configure precision before other JAX imports.
import jax.numpy as jnp
import jax_md
import numpy as np

from ase.io import read
from jax_md import units
from typing import Dict

from so3lr import to_jax_md
from so3lr import So3lrPotential

import time

In [ ]:
# Some helper functions used throughout the example notebook.

# Default nose hoover chain parameters.
def default_nhc_kwargs(
    tau: jnp.float32, 
    overrides: Dict
) -> Dict:
    
    default_kwargs = {
        'chain_length': 3, 
        'chain_steps': 2, 
        'sy_steps': 3,
        'tau': tau
    }
    
    if overrides is None:
        return default_kwargs
  
    return {
      k: overrides.get(k, default_kwargs[k]) for k in default_kwargs
    }

In [ ]:
# Read the periodic water box from the test directory.
path_to_xyz = '../tests/test_data/water_64.xyz'
atoms = read(path_to_xyz, index=-1)
atoms.wrap()

# Repeat in each direction.
atoms = atoms * [2, 2, 2]

species = jnp.array(atoms.get_atomic_numbers())
masses = jnp.array(atoms.get_masses())
num_atoms = len(species)

# Periodic Boundary Conditions

Unlike the `nvt_jaxmd.ipynb` example, this simulation uses periodic boundary conditions (PBCs). `jax_md.space.periodic_general` constructs the displacement and shift functions from the simulation box. A vector `[Lx, Ly, Lz]` represents an orthorhombic cell, while an upper-triangular matrix represents a general triclinic cell. Although the water box is cubic, this NPT example deliberately uses all three box lengths: the barostat must compute a three-dimensional volume, and a one-element vector would instead describe a one-dimensional box. See the [JAX-MD periodic-space documentation](https://jax-md.readthedocs.io/en/main/jax_md.space.html#jax_md.space.periodic_general).

Because the simulation uses fractional coordinates, the initial positions are taken directly from ASE as wrapped scaled coordinates. `jax_md.space.transform` converts them back to Cartesian coordinates when writing or visualizing a frame.

In [ ]:
# Use three box lengths: JAX-MD interprets a one-element vector as a
# one-dimensional volume, which is not valid for this 3D NPT simulation.
cell = np.asarray(atoms.get_cell())
np.testing.assert_allclose(cell, np.diag(np.diag(cell)))
box = jnp.asarray(np.diag(cell), dtype=jnp.float64)
assert np.all(np.asarray(box) > 0)

print('box =', box)

fractional_coordinates = True
displacement, shift = jax_md.space.periodic_general(box=box, fractional_coordinates=fractional_coordinates)

positions_init = jnp.asarray(
    atoms.get_scaled_positions(wrap=True), dtype=jnp.float64
)

# Prepare JAX-MD Energy Function

JAX-MD is built around an energy function and neighbor lists. SO3LR uses separate short- and long-range neighbor lists. The repository's `to_jax_md` convenience interface constructs the energy function and both neighbor-list functions from a `So3lrPotential`, the displacement function, the box, and the species.

In [ ]:
neighbor_fn, neighbor_fn_lr, energy_fn = to_jax_md(
    potential=So3lrPotential(
        model='so3lr-tauri-m-2606',
        dtype=jnp.float64  # or float32 for single precision
    ),
    displacement_or_metric=displacement,
    box_size=box,
    species=species,
    capacity_multiplier=1.25,
    buffer_size_multiplier_sr=1.25,
    buffer_size_multiplier_lr=1.25,
    minimum_cell_size_multiplier_sr=1.0,
    disable_cell_list=False,
    fractional_coordinates=fractional_coordinates
)

# Energy function.
energy_fn = jax.jit(energy_fn)
force_fn = jax.jit(jax_md.quantity.force(energy_fn))

# Initialize the short and long-range neighbor lists.
nbrs = neighbor_fn.allocate(
    positions_init,
    box=box
)
nbrs_lr = neighbor_fn_lr.allocate(
    positions_init,
    box=box
)

In [ ]:
# Check that the energy function is working.
energy_fn(
    positions_init, 
    neighbor=nbrs.idx, 
    neighbor_lr=nbrs_lr.idx, 
    box=box
)

# Structure Relaxation

In [ ]:
# For repeated execution of cell, delete the fire_state first, otherwise jit in jupyter environment 
# can meddle with things.
try:
    del fire_state
except NameError:
    pass

min_cycles = 5
min_steps = 10

fire_init, fire_apply = jax_md.minimize.fire_descent(
    energy_fn, 
    shift, 
    dt_start = 0.05, 
    dt_max = 0.1, 
    n_min = 2
)

fire_apply = jax.jit(fire_apply)
fire_state = fire_init(
    positions_init, 
    box=box, 
    neighbor=nbrs.idx,
    neighbor_lr=nbrs_lr.idx
)

@jax.jit
def step_fire_fn(i, fire_state):
    
    fire_state, nbrs, nbrs_lr = fire_state
    
    fire_state = fire_apply(
        fire_state, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )
    
    nbrs = nbrs.update(
        fire_state.position,
        neighbor=nbrs.idx,
        box=box
    )
    
    nbrs_lr = nbrs_lr.update(
        fire_state.position,
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    return fire_state, nbrs, nbrs_lr

print('Step\tE\tFmax')
print('--------------------------')
for i in range(min_cycles):
    fire_state, nbrs, nbrs_lr = jax.lax.fori_loop(
        0, 
        min_steps, 
        step_fire_fn, 
        (fire_state, nbrs, nbrs_lr)
    )

    # Reject a minimization chunk computed with an overflowed neighbor list.
    if bool(np.asarray(nbrs.did_buffer_overflow)) or bool(
        np.asarray(nbrs_lr.did_buffer_overflow)
    ):
        raise RuntimeError(
            'Neighbor-list capacity overflowed; increase the capacity or buffer multipliers.'
        )
    
    E = energy_fn(
        fire_state.position, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx,
        box=box
    )

    F = force_fn(
        fire_state.position, 
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    print('{}\t{:.2f}\t{:.2f}'.format(i, E, np.abs(F).max()))
    

Print the displacement between the original and optimized positions for the first three atoms. The stored positions are fractional. Use `jax_md.space.transform` for Cartesian coordinates, or the displacement function returned by `jax_md.space.periodic_general` for minimum-image Cartesian displacements.

In [ ]:
print('Difference in fractional coordinates: ')
print((positions_init - fire_state.position)[:3])
print('\n')
print('Difference in real coordinates using transform function: ')
print((jax_md.space.transform(box, positions_init) - jax_md.space.transform(box, fire_state.position))[:3])
print('\n')
print('Difference in real coordinates from displacement function: ')
print((jax.vmap(displacement)(positions_init, fire_state.position))[:3])

# Nose-Hoover Chain NPT simulations

In [ ]:
# Simulation parameters

timestep = 0.0005  # Time step in ps
npt_cycles = 25  # Number of NPT output cycles.
npt_steps = 5  # NPT steps per cycle; total steps are npt_cycles * npt_steps.

T_init = 300  # Initial temperature in K.
pressure_init = 1.01325  # Target pressure in bars. 

chain = 3  # Number of chains in the Nose-Hoover chain.
chain_steps = 2  # Number of steps per chain.
sy_steps = 3

thermo = 100  # Thermostat value in the Nose-Hoover chain. 
baro = 1000  # Barostat value in the Nose-Hoover chain. 

# Dictionary with the NHC settings.
new_nhc_kwargs = {
    'chain_length': chain, 
    'chain_steps': chain_steps, 
    'sy_steps': sy_steps
}

# Convert to metal unit system.
unit = units.metal_unit_system()

timestep = timestep * unit['time']
T_init = T_init * unit['temperature']
pressure = pressure_init * unit['pressure']

rng_key = jax.random.PRNGKey(0)

In [ ]:
# Choose Nose-Hoover thermostat.
init_fn, apply_fn = jax_md.simulate.npt_nose_hoover(
    energy_fn, 
    shift, 
    dt=timestep,
    pressure=pressure, 
    kT=T_init,
    barostat_kwargs=default_nhc_kwargs(baro * timestep, new_nhc_kwargs),
    thermostat_kwargs=default_nhc_kwargs(thermo * timestep, new_nhc_kwargs)
)

apply_fn = jax.jit(apply_fn)
init_fn = jax.jit(init_fn)

# Initialize the state from the relaxed positions and neighbor lists.
state = init_fn(
    rng_key, 
    fire_state.position, 
    box=box, 
    neighbor=nbrs.idx, 
    neighbor_lr=nbrs_lr.idx,
    kT=T_init,
    mass=masses
)

@jax.jit
def step_npt_fn(i, state):
    state, nbrs, nbrs_lr, box = state
    
    state = apply_fn(
        state,
        neighbor=nbrs.idx, 
        neighbor_lr=nbrs_lr.idx, 
        kT=T_init,
        pressure=pressure
    )
    
    box = jax_md.simulate.npt_box(state)
    
    nbrs = nbrs.update(
        state.position, 
        neighbor=nbrs.idx, 
        box=box
    )
    
    nbrs_lr = nbrs_lr.update(
        state.position,
        neighbor_lr=nbrs_lr.idx,
        box=box
    )
    
    return state, nbrs, nbrs_lr, box

# Track total time and step times averaged over cycle.
total_time = time.time()

positions_md = []
box_md = []
temperatures_md = []

print('Step\tKE\tPE\tTot. Energy\tTemp.\tH\ttime/steps\tInvariant drifts (H_i - H_0 , H - H_{i-1}) [meV/atom/ps]')
print('-------------------------------------------------------------------------------------------------------------------------------------')
for i in range(npt_cycles):

    if i == 0:
        initial_H_0 = jax_md.simulate.npt_nose_hoover_invariant(
            energy_fn, 
            state, 
            pressure=pressure,
            kT=T_init,
            neighbor=nbrs.idx, 
            neighbor_lr=nbrs_lr.idx
        )
    
    # Calculate Hamiltonian of the extended NPT dynamics 
    initial_H = jax_md.simulate.npt_nose_hoover_invariant(
        energy_fn, 
        state, 
        pressure=pressure,
        kT=T_init,
        neighbor=nbrs.idx,
        neighbor_lr=nbrs_lr.idx
    )

    old_time = time.time()
    
    # Do `npt_steps` NPT steps.
    new_state, nbrs, nbrs_lr, new_box = jax.block_until_ready(
        jax.lax.fori_loop(
            0, 
            npt_steps, 
            step_npt_fn, 
            (state, nbrs, nbrs_lr, box)
        )
    )
    
    new_time = time.time()
    
    # A chunk computed with an overflowed neighbor list is not valid.
    if bool(np.asarray(nbrs.did_buffer_overflow)) or bool(
        np.asarray(nbrs_lr.did_buffer_overflow)
    ):
        raise RuntimeError(
            'Neighbor-list capacity overflowed; increase the capacity or buffer multipliers.'
        )
    state = new_state
    box = new_box

    # Calculate some quantities for printing
    KE = jax_md.quantity.kinetic_energy(
        momentum=state.momentum,
        mass=state.mass
    )
    
    PE = energy_fn(
        state.position,
        neighbor=nbrs.idx,
        neighbor_lr=nbrs_lr.idx, 
        box=box
    )
    
    T = jax_md.quantity.temperature(
        momentum=state.momentum,
        mass = state.mass
    ) / unit['temperature']
    
    # Calculate initial total energy
    H = jax_md.simulate.npt_nose_hoover_invariant(
        energy_fn,
        state,
        pressure=pressure,
        kT=T_init,
        neighbor=nbrs.idx,
        neighbor_lr=nbrs_lr.idx
    )

    energy_drift_h = (H - initial_H) * 1000 / (timestep / unit['time'] * npt_steps * num_atoms)
    energy_drift_h_0 = (H - initial_H_0) * 1000 / (timestep / unit['time'] * npt_steps * (i + 1) * num_atoms)
    
    positions_md.append(np.array(state.position))
    box_md.append(np.array(box))
    temperatures_md.append(float(T))
    
    print(
        f'{(i+1)*npt_steps}\t{KE:.2f}\t{PE:.2f}\t{KE+PE:.3f}\t{T:.1f}\t{H:.3f}\t{(new_time - old_time) / npt_steps:.2f}\t{energy_drift_h_0:.2f}  ,  {energy_drift_h:.2f}'
    )

assert np.isfinite(np.asarray(positions_md)).all()
assert np.isfinite(np.asarray(box_md)).all()
assert np.all(np.asarray(box_md) > 0)
assert np.isfinite(np.asarray(temperatures_md)).all()
target_kelvin = T_init / unit['temperature']
assert 0.1 * target_kelvin < np.median(temperatures_md) < 10 * target_kelvin

print('Total_time: ', time.time() - total_time)

# Clear all caches
jax.clear_caches()

# Visualization and IO

For this short example, positions and boxes are collected in Python lists, then converted to ASE frames for visualization and a single EXTXYZ write. Each frame retains its contemporaneous cell and periodicity. For larger production runs, use the buffered HDF5 approach in `production_run.ipynb` to avoid making trajectory I/O the simulation bottleneck.

In [ ]:
# If you want to do visualization in the notebook, install nglview by doing `pip install nglview` in your virtualenv
import nglview as nv
from ase import Atoms

atoms_traj = []
for positions, b in zip(positions_md, box_md):
    atoms_traj.append(
        Atoms(
            numbers=np.array(species),
            positions=np.array(jax_md.space.transform(box=b, R=positions)),
            cell=np.diag(np.asarray(b)),
            pbc=True,
        ),
    )

nv.show_asetraj(atoms_traj)

In [ ]:
# Save the frames to xyz.
from ase.io import write

trajectory_path = 'npt_water_md_trajectory.xyz'
write(trajectory_path, atoms_traj, format='extxyz')
reloaded_traj = read(trajectory_path, index=':')
assert len(reloaded_traj) == npt_cycles
for frame, expected_box in zip(reloaded_traj, box_md):
    assert frame.pbc.all()
    np.testing.assert_allclose(frame.cell.lengths(), expected_box)